# Portugal Dry Eye Market — Raw File Automation

End-to-end automation of the IQVIA Santen Portugal raw file into the dashboard-ready tables.

**Pipeline in this notebook**
1. **`raw_df`** — clean the `Final` tab (ACUAISS rename + drop `Total Market`)
2. **`extract_df`** — pivot into the *Portugal_Data_Extract* layout, up to the most recent month, + `Market/Product` (BA) and `Category` (BB)
3. **`Portugal_AT_NonConv`** — replicate the *Country Dashboard Data — Portugal Linked AT Non-Converted* file, columns **A–BL**, same column names & order
4. **`Portugal_AT_Conv`** — copy of NonConv with the unit columns (Q–AN) converted by the ATC4/HU factor `=IFERROR(unit*$F,unit)`

QC cells check record counts and that the unit + revenue **column sums match the Excel** exactly.


In [ ]:
import pandas as pd
import numpy as np
import openpyxl

# ---- File paths (edit if your files live elsewhere) ----
RAW_FILE       = "IQVIA_Santen_Dry_Eye_Market__2026_05.xlsx"
MAPPING_FILE   = "Portugal_Mapping.xlsx"
DASHBOARD_FILE = "Country_Dashboard_Data_-_Portugal_Linked_AT_Non-Converted.xlsx"

RAW_SHEET       = "Final"      # only the 'Final' tab is used from the raw file
MAPPING_SHEET   = "Portugal"   # 'Portugal' tab inside the mapping workbook
DASHBOARD_SHEET = "Data"       # 'Data' tab inside the dashboard workbook

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Step 1 — Build `raw_df`

- **ACUAISS → ACUAISS - AT** (no-op if already done). The hyphen `-` is used deliberately — it is the exact key in the mapping file, so the lookups keep matching.
- **Remove `Total Market`** rows from `Product` (mirrors the Excel `FIND("Total Market",…)` logic). Products merely containing the word *TOTAL* (e.g. `NEOVIS TOTAL MULTI`) are kept.


In [ ]:
raw_df = pd.read_excel(RAW_FILE, sheet_name=RAW_SHEET, header=0)
raw_df["Product"] = raw_df["Product"].astype("string").str.strip()

# 1a) ACUAISS -> ACUAISS - AT
raw_df["Product"] = raw_df["Product"].replace({
    "ACUAISS":      "ACUAISS - AT",
    "ACUAISS – AT": "ACUAISS - AT",   # normalise en-dash variant to the hyphen key
})

# 1b) drop 'Total Market' rows
_before = len(raw_df)
raw_df = raw_df[~raw_df["Product"].str.contains("Total Market", case=False, na=False)].copy()
print(f"'Total Market' rows removed: {_before - len(raw_df)}")
print("raw_df shape:", raw_df.shape, "| columns:", list(raw_df.columns))
raw_df.head()

## Step 2 — Build `extract_df` (the pivot)

Reproduces the *Portugal_Data_Extract* layout: rows = `Geo` / `Product` / `Pack`, then **Sum of Quantity** per month (template cols D–AA) and **Sum of value** per month (cols AB–AY). The month range is dynamic and always runs to the most recent month in `raw_df`. Missing month/product combinations stay `NaN`; an explicit `0` stays `0`.

Quantity columns are prefixed `Qty ` and value columns `Val ` so every name is unique.


In [ ]:
for c in ["Geo", "Product", "Pack"]:
    raw_df[c] = raw_df[c].astype("string").fillna("")

raw_df["dt"]  = pd.to_datetime(raw_df["Date"], format="%b %y")
months        = sorted(raw_df["dt"].unique())
month_labels  = [pd.Timestamp(m).strftime("%b %y") for m in months]
print(f"{len(months)} months  |  first: {month_labels[0]}  ->  most recent: {month_labels[-1]}")

grouped = raw_df.groupby(["Geo", "Product", "Pack", "dt"])[["Quantity", "value"]].sum()
qty = grouped["Quantity"].unstack("dt").reindex(columns=months)
val = grouped["value"].unstack("dt").reindex(columns=months)
qty.columns = [f"Qty {m}" for m in month_labels]   # -> template cols D..AA
val.columns = [f"Val {m}" for m in month_labels]   # -> template cols AB..AY

extract_df = pd.concat([qty, val], axis=1).reset_index()
print("extract_df shape:", extract_df.shape, "| last value column:", extract_df.columns[-1])
extract_df.head()

## Step 3 — Add `Market/Product` (BA) and `Category` (BB)

**BA** → `"Market"` if the product contains `Total Market`, else `"Product"` (normally all `"Product"` after Step 1).
**BB** → VLOOKUP the `Product` in the mapping `Portugal` sheet (match on col C = Product, return col L = Category); `"-"` if not found.


In [ ]:
mapping_cat = pd.read_excel(MAPPING_FILE, sheet_name=MAPPING_SHEET,
                            header=None, skiprows=2, usecols="C,L")
mapping_cat.columns = ["Product", "Category"]
mapping_cat = mapping_cat.dropna(subset=["Product"])
mapping_cat["Product"] = mapping_cat["Product"].astype("string").str.strip()
mapping_cat = mapping_cat.drop_duplicates(subset=["Product"], keep="first")
category_lookup_bb = dict(zip(mapping_cat["Product"], mapping_cat["Category"]))

extract_df["Market/Product"] = np.where(
    extract_df["Product"].str.contains("Total Market", case=False, na=False),
    "Market", "Product")
extract_df["Category"] = extract_df["Product"].map(category_lookup_bb).fillna("-")

print("Market/Product counts:", extract_df["Market/Product"].value_counts().to_dict())
print("Products with no Category match ('-'):",
      extract_df.loc[extract_df["Category"] == "-", "Product"].nunique())
extract_df[["Geo", "Product", "Pack", "Market/Product", "Category"]].head()

## Step 4 — Data-quality check: records with a blank `Product`

Displays every column of any record whose `Product` is blank/empty. Empty table = clean.


In [ ]:
blank_product = extract_df[extract_df["Product"].isna() |
                          (extract_df["Product"].astype("string").str.strip() == "")]
print(f"Blank-Product records: {len(blank_product)}")
blank_product

## Step 5 — Build `Portugal_AT_NonConv` (dashboard columns **A–BL**)

This replicates every column formula from the *Country Dashboard Data — Portugal Linked AT Non-Converted* file, columns **A to BL**, keeping the **exact column names and order** (the header names are read straight from that file).

**How each column is built** (decoded from the Excel formulas):

| Col | Name | Logic |
|---|---|---|
| A | Retina market | `"Inculde"` if *Retina* in Category else `"Exclude"` (source spelling kept) |
| B | Region | constant `"Portugal"` |
| C | RSM | `Product & Pack` |
| D | Territory | mapping `Portugal`: match **Brick** in col AB → return col AD |
| E | Brick | `Geo` from `extract_df` |
| F | ATC4 Class | mapping (key **RSM** in col A) → col J = **HU conversion factor** (this is the `$F` multiplier) |
| G | Category | mapping (key Product in col C) → col L |
| H | Product | `Product` from `extract_df` |
| I | Brand | mapping (Product) → col D |
| J | Company | mapping (Product) → col F |
| K | Pack | mapping (Product) → col G (Pack original) |
| L | Molecule | mapping (Product) → col P |
| M | Sub-Market | mapping (Product) → col M |
| N | Formulation | mapping (key **RSM** in col A) → col O |
| O | Ingredient | mapping (Product) → col Q |
| P | Import | mapping (Product) → col R |
| Q–AN | UN- … (units) | `extract_df` quantity columns, positionally. Market rows subtract the sum of their `"Product"` peers within the same Brick (a no-op while all rows are `"Product"`). Blank → `0`. |
| AO–BL | REV- … (revenue) | `extract_df` value columns, positionally. Blank → `0`. |

> The UN/REV header labels (e.g. `UN-Jan 2021`) are fixed template names in the dashboard; the data is placed **positionally** from the earliest month to the most recent, exactly as the Excel `='Dry Eye Pivot'!…` references do.


In [ ]:
# ---- read the mapping sheet once (no header) and build the lookup blocks ----
mp_raw = pd.read_excel(MAPPING_FILE, sheet_name=MAPPING_SHEET, header=None)

def first_seen(keys, values):
    """VLOOKUP-style: first match wins."""
    out = {}
    for k, v in zip(keys, values):
        if pd.notna(k):
            k = str(k).strip()
            if k not in out:
                out[k] = v
    return out

# A-block: data from row 3 (index 2); key = col A (0)
A_block   = mp_raw.iloc[2:, :18]
atc4_lookup  = first_seen(A_block[0], A_block[9])    # col J  -> HU conversion factor  (F: ATC4 Class)
formu_lookup = first_seen(A_block[0], A_block[14])   # col O  -> Formulation           (N)

# C-block: key = col C (index 2); offset is 1-based within C..R
C_key = mp_raw.iloc[2:, 2]
c_lookup = lambda off: first_seen(C_key, mp_raw.iloc[2:, 2 + off - 1])
category_lookup = c_lookup(10)   # L  Category
brand_lookup    = c_lookup(2)    # D  Brand
company_lookup  = c_lookup(4)    # F  Company
pack_lookup     = c_lookup(5)    # G  Pack original
molecule_lookup = c_lookup(14)   # P  Molecule
submkt_lookup   = c_lookup(11)   # M  Sub-market
ingred_lookup   = c_lookup(15)   # Q  Ingredient
import_lookup   = c_lookup(16)   # R  Parallel import

# AB-block: key = col AB (27) -> Territory col AD (29)
territory_lookup = first_seen(mp_raw.iloc[2:, 27], mp_raw.iloc[2:, 29])

# ---- exact column names / order from the dashboard file ----
_wb = openpyxl.load_workbook(DASHBOARD_FILE, read_only=True)
_hdr = next(_wb[DASHBOARD_SHEET].iter_rows(min_row=1, max_row=1, values_only=True))
_wb.close()
COLS_A_BL   = list(_hdr[:64])     # A..BL
UN_HEADERS  = list(_hdr[16:40])   # Q..AN  (units)
REV_HEADERS = list(_hdr[40:64])   # AO..BL (revenue)

In [ ]:
# ---- helper + base series ----
def VLOOKUP(series, d):
    return series.map(lambda x: d.get(str(x).strip(), "-") if pd.notna(x) else "-")

geo_s  = extract_df["Geo"].astype("string")
prod_s = extract_df["Product"].astype("string")
pack_s = VLOOKUP(prod_s, pack_lookup).astype("string")          # K
rsm_s  = prod_s.fillna("") + pack_s.fillna("")                  # C
cat_s  = VLOOKUP(prod_s, category_lookup)                       # G

# ---- descriptive columns A..P (built by exact header name & order) ----
Portugal_AT_NonConv = pd.DataFrame({
    COLS_A_BL[0]:  np.where(cat_s.astype("string").str.contains("Retina", na=False), "Inculde", "Exclude"),
    COLS_A_BL[1]:  "Portugal",
    COLS_A_BL[2]:  rsm_s,
    COLS_A_BL[3]:  VLOOKUP(geo_s, territory_lookup),
    COLS_A_BL[4]:  geo_s,
    COLS_A_BL[5]:  rsm_s.map(lambda x: atc4_lookup.get(str(x).strip(), "-")),
    COLS_A_BL[6]:  cat_s,
    COLS_A_BL[7]:  prod_s,
    COLS_A_BL[8]:  VLOOKUP(prod_s, brand_lookup),
    COLS_A_BL[9]:  VLOOKUP(prod_s, company_lookup),
    COLS_A_BL[10]: pack_s,
    COLS_A_BL[11]: VLOOKUP(prod_s, molecule_lookup),
    COLS_A_BL[12]: VLOOKUP(prod_s, submkt_lookup),
    COLS_A_BL[13]: rsm_s.map(lambda x: formu_lookup.get(str(x).strip(), "-")),
    COLS_A_BL[14]: VLOOKUP(prod_s, ingred_lookup),
    COLS_A_BL[15]: VLOOKUP(prod_s, import_lookup),
})

# ---- unit columns Q..AN and revenue columns AO..BL (positional) ----
qty_cols = [c for c in extract_df.columns if c.startswith("Qty ")]
val_cols = [c for c in extract_df.columns if c.startswith("Val ")]
assert len(qty_cols) == len(UN_HEADERS),  f"month count {len(qty_cols)} != {len(UN_HEADERS)} UN headers (extends past col BL)"
assert len(val_cols) == len(REV_HEADERS), f"month count {len(val_cols)} != {len(REV_HEADERS)} REV headers"

is_prod = extract_df["Market/Product"].eq("Product").to_numpy()   # Market-residual switch
for un_h, qc in zip(UN_HEADERS, qty_cols):
    q = pd.to_numeric(extract_df[qc], errors="coerce")
    prod_sum_by_geo = q.where(is_prod, 0).groupby(extract_df["Geo"]).transform("sum")
    Portugal_AT_NonConv[un_h] = np.where(is_prod, q, q - prod_sum_by_geo)   # =IF(BA="Product",qty,qty-SUMIFS)
for rev_h, vc in zip(REV_HEADERS, val_cols):
    Portugal_AT_NonConv[rev_h] = pd.to_numeric(extract_df[vc], errors="coerce")

# Excel '=<pivot cell>' turns a blank cell into 0
Portugal_AT_NonConv[UN_HEADERS + REV_HEADERS] = Portugal_AT_NonConv[UN_HEADERS + REV_HEADERS].fillna(0)

# lock exact A..BL order
Portugal_AT_NonConv = Portugal_AT_NonConv[COLS_A_BL]
print("Portugal_AT_NonConv shape:", Portugal_AT_NonConv.shape)
Portugal_AT_NonConv.head()

### QC 1 — record count `extract_df` vs `Portugal_AT_NonConv`

In [ ]:
_n_ext = len(extract_df)
_n_nc  = len(Portugal_AT_NonConv)
print(f"extract_df rows          : {_n_ext}")
print(f"Portugal_AT_NonConv rows : {_n_nc}")
print("MATCH ✅" if _n_ext == _n_nc else "MISMATCH ❌")
assert _n_ext == _n_nc, 'Record counts differ!'

### QC 2 — sum of every **UN** and **REV** column vs the Excel

Order-independent check: the total of each unit/revenue column in `Portugal_AT_NonConv` must equal the same column's total in the dashboard Excel.


In [ ]:
_dash = pd.read_excel(DASHBOARD_FILE, sheet_name=DASHBOARD_SHEET, header=0).iloc[:, :64]

_rows = []
for c in UN_HEADERS + REV_HEADERS:
    s_out = pd.to_numeric(Portugal_AT_NonConv[c], errors="coerce").sum()
    s_xl  = pd.to_numeric(_dash[c], errors="coerce").sum()
    _rows.append({"Column": c, "Sum_NonConv": s_out, "Sum_Excel": s_xl, "Diff": s_out - s_xl})
qc_sum = pd.DataFrame(_rows)

_max_diff = qc_sum["Diff"].abs().max()
print(f"Max absolute sum difference across all UN + REV columns: {_max_diff}")
print("ALL COLUMN SUMS MATCH ✅" if _max_diff < 1e-6 else "SUM MISMATCH ❌ — inspect qc_sum below")
qc_sum

## Step 6 — Build `Portugal_AT_Conv`

A copy of `Portugal_AT_NonConv` where the **unit** columns **Q–AN** are converted using the ATC4 / HU factor (column **F**), replicating the Excel formula that sits in `BN:CK`:

```excel
=IFERROR(<unit cell> * $F2, <unit cell>)
```

i.e. each unit is multiplied by its conversion factor `F`; if `F` isn't numeric, the original unit is kept (the `IFERROR` fallback). The revenue columns **AO–BL** and everything else are unchanged.


In [ ]:
Portugal_AT_Conv = Portugal_AT_NonConv.copy()

conv_factor = pd.to_numeric(Portugal_AT_Conv["ATC4 Class"], errors="coerce")   # $F
for un_h in UN_HEADERS:
    un = pd.to_numeric(Portugal_AT_Conv[un_h], errors="coerce")
    Portugal_AT_Conv[un_h] = np.where(conv_factor.notna(), un * conv_factor, un)  # =IFERROR(un*F, un)

print("Portugal_AT_Conv shape:", Portugal_AT_Conv.shape)
Portugal_AT_Conv.head()

### QC 3 — `Portugal_AT_Conv` sanity

- record count unchanged
- revenue columns identical to NonConv
- units scaled by the conversion factor (shown for one sample month)


In [ ]:
print("row count NonConv == Conv:", len(Portugal_AT_NonConv) == len(Portugal_AT_Conv))

_rev_same = all(
    np.allclose(pd.to_numeric(Portugal_AT_NonConv[c], errors="coerce"),
                pd.to_numeric(Portugal_AT_Conv[c],    errors="coerce"))
    for c in REV_HEADERS)
print("revenue columns unchanged:", _rev_same)

_s = UN_HEADERS[0]
print(f"\nsample unit column '{_s}':")
print(f"  NonConv sum = {pd.to_numeric(Portugal_AT_NonConv[_s], errors='coerce').sum():,.2f}")
print(f"  Conv    sum = {pd.to_numeric(Portugal_AT_Conv[_s],    errors='coerce').sum():,.2f}")

## Step 6.5 — Rename month headers to real months, then fill the 42 period columns (BM–DB)

Runs **before** export, on both `Portugal_AT_NonConv` and `Portugal_AT_Conv`, without changing anything above.

**A. Header rename** — the template labels on `Q:AN` / `AO:BL` are replaced with the real months, ending at `CURRENT_MONTH` (most recent month in the data): `UN-Jun 2024 … UN-May 2026`, `REV-Jun 2024 … REV-May 2026`.

**B. 42 period columns (BM–DB)** — 6 blocks (`Current`, `Previous Period` = −1 month, `Previous Year` = −12 months) × 2 measures (`UN`, `REV`) × 7 variants, aggregating each table's own monthly columns. Fiscal year starts in **April**:

- **single** – that month only
- **Fixed Qtr** – from the fiscal-quarter start up to the month (Q1 Apr–Jun, Q2 Jul–Sep, Q3 Oct–Dec, Q4 Jan–Mar)
- **Rolling Qtr** – the month + previous 2 (3 months)
- **Fixed Tercile** – from the tercile start up to the month (T1 Apr–Jul, T2 Aug–Nov, T3 Dec–Mar)
- **Rolling Tercile** – the month + previous 3 (4 months)
- **FY YTD** – from April up to the month (cumulative fiscal year)
- **MAT** – the month + previous 11 (12 months)


In [ ]:
# ---- CURRENT_MONTH input (leave None to use the most recent month in the data) ----
CURRENT_MONTH = None            # e.g. "May 2026"

month_ts   = [pd.Timestamp(m) for m in months]
current_ts = month_ts[-1] if CURRENT_MONTH is None else pd.Timestamp(pd.to_datetime(CURRENT_MONTH, format="%b %Y"))
print("CURRENT_MONTH used:", current_ts.strftime("%b %Y"))

# ---- A. rename Q:AN and AO:BL to the real months (UN-Mon YYYY / REV-Mon YYYY) ----
real_un  = [f"UN-{t.strftime('%b %Y')}"  for t in month_ts]
real_rev = [f"REV-{t.strftime('%b %Y')}" for t in month_ts]
rename_map = {**dict(zip(UN_HEADERS, real_un)), **dict(zip(REV_HEADERS, real_rev))}
Portugal_AT_NonConv = Portugal_AT_NonConv.rename(columns=rename_map)
Portugal_AT_Conv    = Portugal_AT_Conv.rename(columns=rename_map)

# ---- B. period-aggregation helpers (fiscal year starts in April) ----
def _win_size(A, variant):
    fm = (A.month - 4) % 12 + 1                     # fiscal month: Apr=1 ... Mar=12
    return {"single": 1,
            "Fixed Qtr":       ((fm - 1) % 3) + 1,
            "Rolling Qtr":     3,
            "Fixed Tercile":   ((fm - 1) % 4) + 1,
            "Rolling Tercile": 4,
            "FY YTD":          fm,
            "MAT":             12}[variant]

def _period_sum(df, prefix, A, w):
    cols = [f"{prefix}-{(A - pd.DateOffset(months=k)).strftime('%b %Y')}" for k in range(w)]
    cols = [c for c in cols if c in df.columns]     # only months that exist
    if not cols:
        return pd.Series(0.0, index=df.index)
    return df[cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)

_VARIANTS = ["single", "Fixed Qtr", "Rolling Qtr", "Fixed Tercile", "Rolling Tercile", "FY YTD", "MAT"]
_BLOCKS = [("Current",         current_ts),
           ("Previous Period", current_ts - pd.DateOffset(months=1)),
           ("Previous Year",   current_ts - pd.DateOffset(months=12))]

def add_period_columns(df):
    for measure in ["UN", "REV"]:                   # UN blocks first, then REV  -> BM..DB order
        for block, A in _BLOCKS:
            astr = A.strftime("%b %Y")
            for v in _VARIANTS:
                label = f"{block} {measure}-{astr}" if v == "single" else f"{block} {measure}-{v} {astr}"
                df[label] = _period_sum(df, measure, A, _win_size(A, v))
    return df

Portugal_AT_NonConv = add_period_columns(Portugal_AT_NonConv)
Portugal_AT_Conv    = add_period_columns(Portugal_AT_Conv)

print("Portugal_AT_NonConv shape:", Portugal_AT_NonConv.shape)   # -> 106 columns
print("Portugal_AT_Conv shape:   ", Portugal_AT_Conv.shape)
Portugal_AT_NonConv.head()

## Step 7 — Export the processed output files

Writes both tables to Excel in the same processed format (columns **A–BL**, same names/order), ready to feed the dashboard.

In [ ]:
import os
OUTPUT_DIR = "."   # change to your desired output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)

nonconv_path = os.path.join(OUTPUT_DIR, "Portugal_AT_NonConv.xlsx")
conv_path    = os.path.join(OUTPUT_DIR, "Portugal_AT_Conv.xlsx")

Portugal_AT_NonConv.to_excel(nonconv_path, sheet_name="Data", index=False)
Portugal_AT_Conv.to_excel(conv_path,       sheet_name="Data", index=False)

print("Saved:")
print(" ", nonconv_path, Portugal_AT_NonConv.shape)
print(" ", conv_path,    Portugal_AT_Conv.shape)

## Recap

- **`raw_df`** — cleaned `Final` tab
- **`extract_df`** — pivot extract + `Market/Product` / `Category`
- **`Portugal_AT_NonConv`** — dashboard columns **A–BL**, same names & order; record count ✅ and UN+REV column sums ✅ vs the Excel
- **`Portugal_AT_Conv`** — units (Q–AN) converted by the ATC4/HU factor; revenue untouched

Both tables are built from the same `extract_df`, so they are row-aligned with each other. *(Row order follows the pandas grouping, which can differ slightly from Excel's pivot collation — this doesn't affect the count/sum QCs or downstream joins on keys.)*

Send the next steps whenever you're ready to continue.
